# College Placement Prediction Using Machine Learning
### Final Year BCA Mini-Project & Exploratory Data Analysis Notebook
---
**Objective:** Build and evaluate a complete machine learning classification pipeline to predict student placement status (**Placed** vs **Not Placed**) using academic and skill features.

In [ ]:
# Step 1: Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All libraries loaded successfully!')

In [ ]:
# Step 2: Load the Placement Dataset
df = pd.read_csv('../data/placement_data.csv')
print(f'Total Samples: {df.shape[0]}, Features: {df.shape[1]}')
df.head()

In [ ]:
# Step 3: Dataset Summary and Missing Values Check
df.info()
print('\nMissing values per column:\n', df.isnull().sum())

In [ ]:
# Step 4: Statistical Distribution of Features
df.describe().round(2)

In [ ]:
# Step 5: Placement Target Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Placement_Status', palette=['#3b82f6', '#ef4444'])
plt.title('Placement Status Distribution')
plt.xlabel('Placement Outcome')
plt.ylabel('Count')
plt.show()

In [ ]:
# Step 6: CGPA vs Placement Status
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='Placement_Status', y='CGPA', palette=['#3b82f6', '#ef4444'])
plt.title('CGPA vs Placement Outcome')
plt.show()

In [ ]:
# Step 7: Correlation Heatmap
num_df = df.copy()
if 'Student_ID' in num_df.columns:
    num_df = num_df.drop(columns=['Student_ID'])
num_df['Work_Experience'] = num_df['Work_Experience'].map({'Yes': 1, 'No': 0, '1': 1, '0': 0}).fillna(0).astype(int)
num_df['Placement_Status'] = num_df['Placement_Status'].map({'Placed': 1, 'Not Placed': 0}).fillna(0).astype(int)

plt.figure(figsize=(9, 7))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

In [ ]:
# Step 8: Feature Selection, Stratified Train/Test Split, and Scaling
feature_cols = ['CGPA', 'Tenth_Percentage', 'Twelfth_Percentage', 'Internships', 'Projects', 'Technical_Skills', 'Communication_Skills', 'Backlogs', 'Work_Experience']
X = num_df[feature_cols]
y = num_df['Placement_Status']

# Split: 80% Train, 20% Test (stratified by y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# StandardScaler fit on X_train only to prevent data leakage
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

print(f'Training instances: {X_train.shape[0]}, Test instances: {X_test.shape[0]}')

In [ ]:
# Step 9: Train and Evaluate Multiple Classification Models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1_Score': round(f1, 4)
    })

results_df = pd.DataFrame(results)
print('=== MODEL COMPARISON TABLE ===')
print(results_df.to_string(index=False))

In [ ]:
# Step 10: Best Model Selection and Conclusion
best_row = results_df.sort_values(by=['F1_Score', 'Accuracy'], ascending=False).iloc[0]
print(f"Selected Model: {best_row['Model']}")
print(f"Test Accuracy:  {best_row['Accuracy']*100:.2f}%")
print(f"F1-Score:       {best_row['F1_Score']}")